# Model Training & Batch Transform DAG
This is the core ML pipeline. We pull data from our **Feature Store**, train a Random Forest on SageMaker infrastructure, and execute a **Batch Transform** job to generate predictions.

In [7]:
!pip install "sagemaker<3.0.0" awswrangler

import os
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import awswrangler as wr
import time

# Setup
role = sagemaker.get_execution_role()
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "aai-540-group6/nyc-collisions-ml"

print(f"Executing as Role: {role}")

Executing as Role: arn:aws:iam::594126158441:role/LabRole


### 1. Retrieve Data from Feature Store
Run an Athena query to grab the engineered features from our Offline Store.

In [8]:
# feature group name
fg_name = "collisions-07-05-51"

fg = FeatureGroup(name=fg_name, sagemaker_session=sess)
query = fg.athena_query()
db = query.database
tbl = query.table_name

sql = f'SELECT * FROM "{db}"."{tbl}"'
print(f"Pulling data via: {sql}")

df = wr.athena.read_sql_query(sql, database=db)
print(f"Retrieved {df.shape[0]} rows.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pulling data via: SELECT * FROM "sagemaker_featurestore"."collisions_07_05_51_1780811507"


INFO:awswrangler.athena._utils:Created CTAS table "sagemaker_featurestore"."temp_table_e43118f2fbff4dd7a217930114e0cd57"


Retrieved 50000 rows.


### 2. Prepare the Splits
We need a 3-way split: Training, Validation, and a Batch inference set (without the target label).

In [9]:
features = ['borough', 'month', 'hour', 'is_rush_hour', 'is_weekend', 'cause_category', 'vehicle_type']
target = 'target'
id_col = 'collision_id'

# Random split
rand = np.random.rand(len(df))
train_idx = rand < 0.8
val_idx = (rand >= 0.8) & (rand < 0.9)
batch_idx = rand >= 0.9

df_train = df[train_idx][features + [target]]
df_val = df[val_idx][features + [target]]
df_batch = df[batch_idx][features + [id_col]] # IDs kept for joining

print(f"Splits -> Train: {len(df_train)}, Val: {len(df_val)}, Batch: {len(df_batch)}")

Splits -> Train: 40068, Val: 4926, Batch: 5006


### 3. Upload to S3

In [10]:
os.makedirs('data_splits', exist_ok=True)
df_train.to_csv('data_splits/train.csv', index=False, header=False)
df_val.to_csv('data_splits/val.csv', index=False, header=False)
df_batch.to_csv('data_splits/batch.csv', index=False, header=False)

s3_train = sess.upload_data('data_splits/train.csv', bucket, f"{prefix}/train")
s3_val = sess.upload_data('data_splits/val.csv', bucket, f"{prefix}/validation")
s3_batch = sess.upload_data('data_splits/batch.csv', bucket, f"{prefix}/batch")

### 4. Training (SKLearn Estimator)
Kick off a training job on a dedicated ml.m5.xlarge instance.

In [11]:
from sagemaker.sklearn.estimator import SKLearn

est = SKLearn(
    entry_point='sagemaker_train.py',
    source_dir='../src',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters={'n-estimators': 100, 'max-depth': 12}
)

est.fit({'train': s3_train})
print("Model fit complete.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-06-07-06-10-03-097


2026-06-07 06:10:06 Starting - Starting the training job.

.

.

...


2026-06-07 06:11:48 Training - Training image download completed. Training in progress./miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-07 06:11:58,140 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-06-07 06:11:58,144 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-07 06:11:58,146 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-07 06:11:58,160 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-06-07 06:11:58,520 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 128.1 MB/s  0:00:00
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Uninstalling python-dateutil-2.8.1:
      Successfully uninstalled python-dateutil-2.8.1
  Attempting uninstall: pluggy
    Found existing installation: pluggy 1.0.0
    Uninstalling pluggy-1.0.0:
      Successfully uninstalled pluggy-1.0.0
  Attempting uninstall: pandas
    Found existing installation: pandas 1.1.3
    Uninstalling pandas-1.1.3:
      Successfully uninstalled pandas-1.1.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker-sklearn-container 2.0 requires pandas==1.1.3, but you have pandas 2.3.3 which is incompatible.
sagemaker-sklearn-container 2.0 requires python-dateutil==2.8.1, but you have python-dateutil 2.9.0.post0 which is incompatible.
[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
2026-06-07 06:12:20,958 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-07 06:12:20,961 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-07 06:12:20,979 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-07 06:12:20,982 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-07 06:12:21,000 sagemaker-trainin


2026-06-07 06:12:37 Uploading - Uploading generated training model
2026-06-07 06:12:37 Failed - Training job failed


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:14                                                                                   │
│                                                                                                  │
│   11 │   hyperparameters={'n-estimators': 100, 'max-depth': 12}                                  │
│   12 )                                                                                           │
│   13                                                                                             │
│ ❱ 14 est.fit({'train': s3_train})                                                                │
│   15 print("Model fit complete.")                                                                │
│   16                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:171 in wrapper  │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:142 in wrapper  │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                            

### 5. Batch Inferences (I/O Joining)
Execute the transform job and join predictions with their IDs.

In [ ]:
trans = est.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f"s3://{bucket}/{prefix}/predictions",
    assemble_with='Line'
)

trans.transform(
    data=s3_batch,
    content_type='text/csv', 
    split_type='Line',
    input_filter='$[1:]', # Send only features to model
    join_source='Input',
    output_filter='$[0,-1]' # Output ID and result
)

trans.wait()
print("Predictions ready in S3.")

### 6. Final Audit
Download and check the results.

In [ ]:
res = wr.s3.read_csv(f"s3://{bucket}/{prefix}/predictions/batch.csv.out", header=None)
res.columns = ['id', 'pred']
display(res.head())